In [1]:
import json
import os

from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite',
    temperature = 0,
    google_api_key=os.getenv('GOOGLE_API_KEY')
)

In [5]:
categories = [
    "Math",
    "Music",
    "Programming",
]

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a text classification assistant.

Your task is to classify the user's message into ONLY ONE category.

Allowed categories:
{categories}

Return ONLY valid JSON.

Output format:
{{
    "category_name": "<category>",
    "score": 0.0
}}

Rules:
- category_name must be one of the allowed categories.
- score must be between 0 and 1.
- Do not explain anything.
- Do not use markdown.
- Return JSON only.
""",
        ),
        (
            "human",
            "{message}",
        ),
    ]
)

In [7]:
output_parser =  StrOutputParser()

In [29]:
chain = prompt | llm | output_parser

In [13]:
message = "How do I write a Python function?"

In [14]:
result = chain.invoke(
    {
        "message": message,
        "categories": ", ".join(categories),
    }
)

In [17]:
print(result), result

{
"category_name": "Programming",
"score": 1.0
}


(None, '{\n"category_name": "Programming",\n"score": 1.0\n}')

In [18]:
data = json.loads(result)

In [19]:
data

{'category_name': 'Programming', 'score': 1.0}

In [20]:
from pydantic import BaseModel, Field

In [21]:
from pydantic import BaseModel, Field

class Classification(BaseModel):
    category_name: str = Field(
        description="Predicted category. Must be Math, Music or Programming."
    )

    score: float = Field(
        description="Confidence score between 0 and 1."
    )

In [22]:
structured_llm = llm.with_structured_output(Classification)

chain = prompt | structured_llm

In [24]:
result = chain.invoke({
    'message': message,
    'categories': ', '.join(categories)
})

In [26]:
result.category_name

'Programming'

In [30]:
for chunk in chain.stream(
    {
        "message": message,
        "categories": ", ".join(categories),
    }
):
    print(chunk, end="", flush=True)

{
"category_name": "Programming",
"score": 1.0
}